# 6 · ¿Ayudan las features agronómicas de dominio?

El **panel unificado ya incluye clima + NDVI-AVHRR (verdor) + ERA5-Land (suelo,
heladas)**: el análisis histórico mostró que el estado de suelo + verdor **corren el techo
un poco** (mejora chica pero real sobre el clima solo), así que quedaron como parte del
dataset por defecto.

La pregunta abierta acá es la que sigue viva: sobre ese panel, ¿aportan además las
**features agronómicas de dominio** —balance hídrico, estrés térmico de la ventana
crítica (`add_agro_features`)— algo que el modelo no saque ya de las columnas mensuales?

> Dirección futura (a diseñar): feature engineering más profundo —índices de extremos
> (días Tmax>32 en floración, racha seca máxima, déficit hídrico acumulado), NDVI integrado
> de temporada (≈ biomasa) y anomalías de NDVI vs su climatología, interacciones
> suelo×precip—, en vez de solo cuatro features agregadas.

In [1]:
import lab                      # utilidades: métricas y gráficos (experiments/lab.py)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data
from src.models import (IsolationForestDetector, OneClassSVMDetector,
                        ZScoreDetector, MahalanobisDetector,
                        AEDetector, DenoisingAEDetector, VAEDetector,
                        EnsembleDetector, DeepODDetector)
plt.rcParams["figure.dpi"] = 110
SEEDS = lab.SEEDS            # 10 semillas del estudio. Bajalas (p. ej. [42,43,44])
print("semillas:", SEEDS)   # para un Run all más rápido; los números se mueven ±std

semillas: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


## 6.1 · La sonda: 3 detectores multi-seed sobre un dataset
Misma evaluación en igualdad de condiciones (5 semillas, media ± std) para el VAE
`recon_prob`, IForest y OCSVM-RBF.

In [2]:
SEEDS_F = SEEDS[:5]
def probe(dset):
    y = dset.y_test
    cols = {"VAE recon_prob": [], "IForest": [], "OCSVM-RBF": []}
    for s in SEEDS_F:
        v = VAEDetector(hidden_dims=(128,64), latent_dim=24, score_mode="recon_prob",
                        n_mc_samples=50, max_epochs=300, patience=30, random_state=s).fit(dset.X_train)
        cols["VAE recon_prob"].append(lab.metrics(v.score_samples(dset.X_test), y)["pr_auc"])
        cols["IForest"].append(lab.metrics(IsolationForestDetector(n_estimators=200, max_features=0.3,
            random_state=s).fit(dset.X_train).score_samples(dset.X_test), y)["pr_auc"])
        cols["OCSVM-RBF"].append(lab.metrics(OneClassSVMDetector(kernel="rbf", nu=0.1)
            .fit(dset.X_train).score_samples(dset.X_test), y)["pr_auc"])
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in cols.items()}

## 6.2 · Dataset unificado vs. unificado + agro

In [3]:
panel_z = data.prepare()                              # panel unificado (clima+NDVI+ERA5)
ds_uni  = data.build_crop_dataset(panel_z, "soja")
ds_agro = data.build_crop_dataset(panel_z, "soja", use_agro=True)
for nombre, d in [("unificado", ds_uni), ("+ agro", ds_agro)]:
    print(f"{nombre:12s} {len(d.feature_cols):3d} features")

[data] dedup panel: 27862 -> 20755 filas (7107 duplicados espurios por lat/lon eliminados)


unificado     65 features
+ agro        69 features


## 6.3 · Resultados (PR-AUC test ± std, soja)

In [4]:
res = {"unificado": probe(ds_uni), "+ agro": probe(ds_agro)}
pd.DataFrame({n: {k: f"{m:.3f}±{s:.3f}" for k,(m,s) in r.items()} for n,r in res.items()}).T

,VAE recon_prob,IForest,OCSVM-RBF
unificado,0.583±0.044,0.552±0.017,0.506±0.000
+ agro,0.565±0.052,0.572±0.010,0.515±0.000


## 6.4 · La prueba justa: seed-ensemble unificado vs. + agro
El modelo final es el **seed-ensemble** (nb 4), que baja fuerte el ±std, así que una mejora
chica se puede ver ahí mejor que en un VAE single:

In [5]:
def ens_scores(dset, base):
    miembros = [VAEDetector(hidden_dims=(128,64), latent_dim=24, score_mode="recon_prob",
                n_mc_samples=50, max_epochs=300, patience=30, random_state=base+i) for i in range(10)]
    return EnsembleDetector(miembros).fit(dset.X_train).score_samples(dset.X_test)

for nombre, dset in [("ensemble unificado", ds_uni), ("ensemble + agro", ds_agro)]:
    prs = [lab.metrics(ens_scores(dset, b), dset.y_test)["pr_auc"] for b in [42, 52, 62]]
    print(f"{nombre:22s} {np.mean(prs):.3f}±{np.std(prs):.3f}")

ensemble unificado     0.619±0.015


ensemble + agro        0.596±0.017


**Conclusión:** el panel unificado (con suelo + verdor) es la base del modelo final.
Las features agronómicas agregadas no mueven la aguja de forma clara sobre eso (comparar los
±std de arriba) — coherente con el techo estructural (nb 5): las cuatro agro son
transformaciones de las mismas columnas mensuales que el modelo ya ve. Exprimir más señal
requiere **feature engineering de dominio más rico** (ver la nota del intro), no solo cuatro
agregados — es la dirección de trabajo pendiente.